# 12 — Sprint 4: fundamentos da análise comercial

Este notebook concentra somente o vocabulário técnico compartilhado, a normalização não destrutiva e o acesso à base de conhecimento. Ele não executa uma análise sozinho; prepara as funções pequenas que os notebooks seguintes reutilizam no mesmo kernel.


## Contrato comum

A transcrição original nunca é modificada. Uma cópia normalizada é usada apenas para comparação lexical. Os modos aceitos são `auto`, `full` e `fallback`, e todas as dependências pesadas serão carregadas sob demanda nos módulos especializados.

Execute este notebook antes dos módulos 13 a 17, ou abra o notebook 18 para carregá-los em sequência. As funções aqui definidas preparam cópias para busca e inferência; o texto recebido permanece íntegro no contrato final.


In [ ]:
from __future__ import annotations

from collections import Counter

from functools import lru_cache

import json

import math

from pathlib import Path

import re

from typing import Any

import unicodedata

SUPPORTED_MODES = {"auto", "full", "fallback"}

PORTUGUESE_STOPWORDS = {
    "a", "ao", "aos", "as", "com", "como", "da", "das",
    "de", "do", "dos", "e", "em", "entre", "essa", "esse",
    "esta", "este", "foi", "isso", "mais", "mas", "muito",
    "na", "nao", "nas", "no", "nos", "o", "os", "ou",
    "para", "pela", "pelo", "por", "que", "se", "sem", "ser",
    "sua", "sao", "tem", "um", "uma", "voce", "cliente",
    "reuniao", "atual", "cenario", "acompanhamento",
}

## Preparação interna do texto

Os chunks por palavras limitam o tamanho enviado aos modelos. A normalização remove diferenças de caixa e acentuação apenas da cópia de trabalho; `_tokens` também descarta stopwords para o ranking lexical.

A divisão em palavras controla o tamanho dos trechos enviados a sentimento e churn. Ela é diferente do chunking por tokenizer do experimento da Sprint 3; cada indicador documenta o limite que usa.


In [ ]:
def _word_chunks(text: str, max_words: int = 80) -> list[str]:
    words = text.split()
    return [" ".join(words[start:start + max_words]) for start in range(0, len(words), max_words)]


### Normalizar a cópia de trabalho

A comparação lexical usa uma cópia em minúsculas e sem acentos. Esse valor nunca substitui `transcricao_original` no resultado.


In [ ]:
def _normalize(text: str) -> str:
    normalized = unicodedata.normalize("NFKD", str(text).casefold())
    return "".join(char for char in normalized if not unicodedata.combining(char))


### Extrair tokens lexicais

A tokenização simples atende apenas ao BM25 e à lista de termos. Stopwords são descartadas aqui, sem remover palavras da transcrição enviada aos modelos.


In [ ]:
def _tokens(text: str) -> list[str]:
    return [
        token
        for token in re.findall(r"[a-z0-9+]+", _normalize(text))
        if len(token) >= 2 and token not in PORTUGUESE_STOPWORDS
    ]


## Motivo seguro do fallback

Quando um modelo não pode ser carregado no modo `auto`, registramos uma causa curta e auditável. O texto original da exceção não é exposto, pois pode conter caminhos locais ou outros detalhes do ambiente.

O campo `code` categoriza a falha e `error_type` informa sua classe. O modo `full` propaga o erro; apenas `auto` usa essa causa para trocar de mecanismo.


In [ ]:
def _fallback_reason(error: Exception) -> dict[str, str]:
    current = error
    visited: set[int] = set()
    while current.__cause__ is not None and id(current) not in visited:
        visited.add(id(current))
        current = current.__cause__

    if isinstance(current, (ModuleNotFoundError, ImportError)):
        code = "missing_dependency"
    elif isinstance(current, FileNotFoundError):
        code = "missing_artifact"
    elif isinstance(current, ValueError):
        code = "incompatible_artifact"
    elif isinstance(current, OSError):
        code = "artifact_unavailable"
    else:
        code = "model_load_failed"

    return {"code": code, "error_type": type(current).__name__}


## Localização dos artefatos

A raiz é descoberta a partir do diretório atual ou de `PROJECT_ROOT`, usado pelos testes. O catálogo TOTVS e seus aliases são lidos uma vez e mantidos em cache durante a sessão.

Se estiver na raiz ou em `notebooks/`, a busca encontra `data/knowledge_base`. `PROJECT_ROOT` permite execução controlada nos testes. Falhas de localização aparecem antes da consulta.


In [ ]:
def _project_root() -> Path:
    configured = globals().get("PROJECT_ROOT")
    if configured is not None:
        root = Path(configured).resolve()
        if (root / "data" / "knowledge_base").exists():
            return root
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / "knowledge_base").exists():
            return candidate
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto Wedjat.")


### Carregar o catálogo TOTVS

A função localiza dois arquivos da base versionada: documentos TOTVS e grupos de aliases. Ela devolve a lista de documentos e a lista de grupos, usadas no ranking de produtos e na extração de termos. A leitura não modifica os arquivos nem a transcrição.


In [ ]:
def _load_catalog() -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    root = _project_root()
    knowledge_base = json.loads(
        (root / "data" / "knowledge_base" / "totvs_rag_kb_v1.json").read_text(
            encoding="utf-8"
        )
    )
    aliases_payload = json.loads(
        (root / "data" / "knowledge_base" / "rag_aliases.json").read_text(
            encoding="utf-8"
        )
    )
    return knowledge_base, aliases_payload["groups"]
